# 03 — Thai Transcript Extraction จากวิดีโอ

Notebook นี้ต่อจากขั้น Face Detection + Mouth Extraction

เป้าหมายคือสร้าง **ข้อความภาษาไทย + timestamp** จากวิดีโอ เพื่อเตรียมทำ label สำหรับ Visual Speech Recognition (Lip Reading)

สิ่งที่ได้:
- `transcript_segments.csv`
- `transcript_words.csv`
- `transcript_with_frames.csv`
- `transcript.txt`

> transcript จาก Speech-to-Text เป็น **weak label** ควรตรวจแก้ก่อนใช้ train supervised model จริง

## Cell 1 — ตั้ง Cache ไป Drive D

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
CACHE_ROOT = PROJECT_ROOT / "cache"

HF_HOME = CACHE_ROOT / "huggingface"
TORCH_HOME = CACHE_ROOT / "torch"
TEMP_HOME = CACHE_ROOT / "temp"

for p in [HF_HOME, TORCH_HOME, TEMP_HOME]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TORCH_HOME"] = str(TORCH_HOME)
os.environ["TEMP"] = str(TEMP_HOME)
os.environ["TMP"] = str(TEMP_HOME)

print("HF_HOME   :", HF_HOME)
print("TORCH_HOME:", TORCH_HOME)
print("TEMP      :", TEMP_HOME)

## Cell 2 — ตรวจ Python / CUDA

In [ ]:
import sys
import torch

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)
print("Python :", sys.executable)
print("Version:", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("GPU    : ไม่พบ CUDA — ใช้ CPU ได้แต่จะช้ากว่า")

## Cell 3 — ติดตั้ง packages

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "faster-whisper",
    "pandas",
    "tqdm"
])

print("Dependencies ready")

## Cell 4 — ตรวจ FFmpeg

In [ ]:
import shutil

ffmpeg_path = shutil.which("ffmpeg")

if ffmpeg_path:
    print("System FFmpeg found:", ffmpeg_path)
else:
    print("System FFmpeg not found — this is OK.")
    print("faster-whisper will use PyAV for decoding.")

## Cell 5 — เลือกวิดีโอ

In [ ]:
VIDEO_ID = "video001"

RAW_VIDEO_DIR = PROJECT_ROOT / "dataset" / "raw_videos"
VIDEO_PATH = RAW_VIDEO_DIR / f"{VIDEO_ID}.webm"

TRANSCRIPT_ROOT = PROJECT_ROOT / "dataset" / "transcripts" / VIDEO_ID
TRANSCRIPT_ROOT.mkdir(parents=True, exist_ok=True)

print("VIDEO_PATH      :", VIDEO_PATH)
print("TRANSCRIPT_ROOT :", TRANSCRIPT_ROOT)
print("Video exists    :", VIDEO_PATH.exists())

if not VIDEO_PATH.exists():
    raise FileNotFoundError(VIDEO_PATH)

## Cell 6 — อ่าน FPS ของวิดีโอ

In [ ]:
import cv2

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(f"เปิดวิดีโอไม่ได้: {VIDEO_PATH}")

FPS = float(cap.get(cv2.CAP_PROP_FPS))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

DURATION = TOTAL_FRAMES / FPS if FPS > 0 else 0

print("FPS         :", FPS)
print("Frames      :", TOTAL_FRAMES)
print("Resolution  :", f"{WIDTH}x{HEIGHT}")
print("Duration(s) :", round(DURATION, 2))

## Cell 7 — โหลด Whisper Model

In [ ]:
from faster_whisper import WhisperModel

MODEL_SIZE = "medium"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("Model       :", MODEL_SIZE)
print("Device      :", DEVICE)
print("Compute type:", COMPUTE_TYPE)

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    download_root=str(HF_HOME / "faster_whisper")
)

print("Whisper model ready")

## Cell 8 — ถอดเสียงภาษาไทย

In [ ]:
segments_generator, info = model.transcribe(
    str(VIDEO_PATH),
    language="th",
    beam_size=5,
    vad_filter=True,
    vad_parameters=dict(
        min_silence_duration_ms=400
    ),
    word_timestamps=True,
    condition_on_previous_text=True
)

print("Detected language:", info.language)
print("Language probability:", getattr(info, "language_probability", None))

segments = list(segments_generator)

print("Segments:", len(segments))

## Cell 9 — แปลงผลเป็น DataFrame

In [ ]:
import pandas as pd
import math

segment_rows = []
word_rows = []

for seg_index, seg in enumerate(segments):
    text = (seg.text or "").strip()

    segment_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": seg_index,
        "start_time": round(float(seg.start), 3),
        "end_time": round(float(seg.end), 3),
        "duration": round(float(seg.end - seg.start), 3),
        "text": text,
    })

    if seg.words:
        for word_index, word in enumerate(seg.words):
            word_rows.append({
                "video_id": VIDEO_ID,
                "segment_id": seg_index,
                "word_index": word_index,
                "start_time": round(float(word.start), 3),
                "end_time": round(float(word.end), 3),
                "word": (word.word or "").strip(),
                "probability": (
                    round(float(word.probability), 6)
                    if word.probability is not None
                    else None
                ),
            })

segments_df = pd.DataFrame(segment_rows)
words_df = pd.DataFrame(word_rows)

print("Segment rows:", len(segments_df))
print("Word rows   :", len(words_df))
display(segments_df.head(20))

## Cell 10 — ทำความสะอาดข้อความ

In [ ]:
import re

def clean_thai_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

segments_df["text_raw"] = segments_df["text"]
segments_df["text"] = segments_df["text"].apply(clean_thai_text)

if len(words_df) > 0:
    words_df["word_raw"] = words_df["word"]
    words_df["word"] = words_df["word"].apply(clean_thai_text)

display(segments_df.head(20))

## Cell 11 — แปลง Timestamp เป็น Frame Index

In [ ]:
segments_df["start_frame"] = (
    segments_df["start_time"] * FPS
).apply(math.floor).astype(int)

segments_df["end_frame"] = (
    segments_df["end_time"] * FPS
).apply(math.ceil).astype(int)

segments_df["frame_count"] = (
    segments_df["end_frame"]
    - segments_df["start_frame"]
    + 1
)

if len(words_df) > 0:
    words_df["start_frame"] = (
        words_df["start_time"] * FPS
    ).apply(math.floor).astype(int)

    words_df["end_frame"] = (
        words_df["end_time"] * FPS
    ).apply(math.ceil).astype(int)

display(
    segments_df[
        [
            "segment_id",
            "start_time",
            "end_time",
            "start_frame",
            "end_frame",
            "frame_count",
            "text",
        ]
    ].head(20)
)

## Cell 12 — Export CSV / TXT

In [ ]:
SEGMENTS_CSV = TRANSCRIPT_ROOT / "transcript_segments.csv"
WORDS_CSV = TRANSCRIPT_ROOT / "transcript_words.csv"
FRAMES_CSV = TRANSCRIPT_ROOT / "transcript_with_frames.csv"
TXT_PATH = TRANSCRIPT_ROOT / "transcript.txt"

segments_df.to_csv(
    SEGMENTS_CSV,
    index=False,
    encoding="utf-8-sig"
)

words_df.to_csv(
    WORDS_CSV,
    index=False,
    encoding="utf-8-sig"
)

segments_df[
    [
        "video_id",
        "segment_id",
        "start_time",
        "end_time",
        "start_frame",
        "end_frame",
        "frame_count",
        "text",
    ]
].to_csv(
    FRAMES_CSV,
    index=False,
    encoding="utf-8-sig"
)

with open(TXT_PATH, "w", encoding="utf-8") as f:
    for _, row in segments_df.iterrows():
        f.write(
            f"[{row['start_time']:.3f} - "
            f"{row['end_time']:.3f}] "
            f"{row['text']}\n"
        )

print("Saved:")
print(" -", SEGMENTS_CSV)
print(" -", WORDS_CSV)
print(" -", FRAMES_CSV)
print(" -", TXT_PATH)

## Cell 13 — Preview Transcript

In [ ]:
for _, row in segments_df.head(50).iterrows():
    print(
        f"[{row['start_time']:8.3f} → "
        f"{row['end_time']:8.3f}] "
        f"{row['text']}"
    )

## Cell 14 — ตรวจ overlap กับ Mouth Metadata

In [ ]:
MOUTH_METADATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "mouth_crops"
    / VIDEO_ID
    / "mouth_metadata.csv"
)

print("Mouth metadata exists:", MOUTH_METADATA_PATH.exists())

overlap_rows = []

if MOUTH_METADATA_PATH.exists():

    mouth_df = pd.read_csv(MOUTH_METADATA_PATH)
    mouth_df = mouth_df[
        mouth_df["accepted"] == True
    ].copy()

    for _, seg in segments_df.iterrows():

        matched = mouth_df[
            (mouth_df["frame_index"] >= seg["start_frame"])
            &
            (mouth_df["frame_index"] <= seg["end_frame"])
        ]

        counts = (
            matched
            .groupby("track_id")["frame_index"]
            .count()
            .to_dict()
        )

        overlap_rows.append({
            "segment_id": int(seg["segment_id"]),
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "start_frame": int(seg["start_frame"]),
            "end_frame": int(seg["end_frame"]),
            "text": seg["text"],
            "mouth_frames_by_track": counts,
        })

    overlap_df = pd.DataFrame(overlap_rows)
    display(overlap_df.head(30))

else:
    print("ยังไม่มี mouth_metadata.csv")

## Cell 15 — Export Transcript ↔ Mouth Overlap

In [ ]:
if "overlap_df" in globals() and len(overlap_df) > 0:

    OVERLAP_CSV = (
        TRANSCRIPT_ROOT
        / "transcript_mouth_overlap.csv"
    )

    overlap_df.to_csv(
        OVERLAP_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", OVERLAP_CSV)

else:
    print("ยังไม่มี overlap data")

# ขั้นต่อไป

หลัง Notebook นี้ผ่าน เราจะมี:

```text
video001.webm
├── Mouth 96×96
└── Transcript + timestamp
```

ขั้นถัดไปคือ **Speaker / Track Alignment** เพื่อหาว่า transcript แต่ละช่วงเป็นของ `track_000` หรือ `track_001`

จากนั้นจึงสร้าง labeled lip clips สำหรับ train